# Chat history, context windows, and memory

## Learning goals

- Represent conversation roles and complete turns
- Distinguish chat history, model context, and durable memory
- Build a bounded context window without losing system instructions
- Understand truncation, summarization, retrieval, and their trade-offs
- Test context construction without calling a model


## Three ideas that are easy to confuse

- **Chat history** is the record your application has collected: user messages, assistant replies, and possibly tool calls and observations.
- **Context** is the bounded input sent to the model for one request. It may contain only a selected portion of the history plus instructions and retrieved information.
- **Memory** is information persisted beyond the current context window or session. It might live in a database, profile store, summary, or retrieval index.

A model does not automatically remember an earlier API call. Your application must either send the relevant information again or use a provider feature that manages conversation state. Even then, the application still owns retention, privacy, and relevance decisions.


## Mental model

```mermaid
flowchart LR
  A[Durable conversation history] --> B[Context builder]
  C[System or developer instructions] --> B
  D[Retrieved relevant memory] --> B
  E[Newest user message] --> B
  B -->|select, order, trim| F[Bounded model context]
  F --> G[Model call]
  G --> H[Assistant response]
  H --> A
```

The context builder is the important application component. It decides what the model can see *for this turn*. A sensible builder normally preserves high-priority instructions, includes the newest user request, adds recent complete exchanges, and inserts only relevant durable memory.

> More context is not automatically better. Irrelevant history increases cost, latency, distraction, and the chance that stale instructions influence the answer.


## Represent messages explicitly

Roles are part of the protocol, not decoration. This teaching model uses `system`, `user`, and `assistant`. Tool-calling notebooks will later add tool-call identifiers and tool observations.


In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class Message(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str = Field(min_length=1)


system_message = Message(
    role="system",
    content="You are a concise travel-planning assistant.",
)
print(system_message.model_dump())


## Build context by complete exchanges

A context budget must reserve room for the model's answer. If a model supports a total context window of `N` tokens, the input cannot safely consume all `N`; instructions, tool schemas, and expected output need space too.

For this offline example, we use a rough character-based estimate. Production code should use the tokenizer or usage estimator documented for the selected model. The algorithm keeps user/assistant exchanges together so trimming does not leave a confusing orphaned assistant reply.


In [ ]:
def estimate_tokens(text: str) -> int:
    # Teaching heuristic only: many English texts average roughly 4 chars/token.
    return max(1, (len(text) + 3) // 4)


def message_cost(message: Message) -> int:
    # Include a small allowance for role and message-structure overhead.
    return estimate_tokens(message.content) + 4


class Conversation:
    def __init__(self, system: Message):
        if system.role != "system":
            raise ValueError("The first message must be a system message.")
        self.system = system
        self.exchanges: list[tuple[Message, Message]] = []

    def add_exchange(self, user_text: str, assistant_text: str) -> None:
        self.exchanges.append((
            Message(role="user", content=user_text),
            Message(role="assistant", content=assistant_text),
        ))

    def build_context(self, newest_user_text: str, max_input_tokens: int) -> list[Message]:
        newest = Message(role="user", content=newest_user_text)
        required = [self.system, newest]
        used = sum(message_cost(message) for message in required)
        if used > max_input_tokens:
            raise ValueError("System instructions and newest request exceed the input budget.")

        selected: list[tuple[Message, Message]] = []
        for exchange in reversed(self.exchanges):
            exchange_cost = sum(message_cost(message) for message in exchange)
            if used + exchange_cost > max_input_tokens:
                break
            selected.append(exchange)
            used += exchange_cost

        context = [self.system]
        for exchange in reversed(selected):
            context.extend(exchange)
        context.append(newest)
        return context


In [ ]:
conversation = Conversation(system_message)
conversation.add_exchange(
    "I am planning a trip to Rajasthan.",
    "Great—what dates and interests should I plan around?",
)
conversation.add_exchange(
    "I like architecture and vegetarian food.",
    "I will prioritize forts, palaces, and vegetarian restaurants.",
)
conversation.add_exchange(
    "Keep the pace relaxed.",
    "Understood: fewer stops and more time at each place.",
)

context = conversation.build_context(
    "Plan two days in Jaipur.",
    max_input_tokens=70,
)

for message in context:
    print(f"{message.role:>9}: {message.content}")
print("Estimated input tokens:", sum(message_cost(m) for m in context))


## Four context strategies

| Strategy | What it does | Strength | Main risk |
|---|---|---|---|
| Sliding window | Keeps the latest complete turns | Simple and faithful to recent wording | Drops older facts abruptly |
| Summarization | Replaces older turns with a compact summary | Retains more history cheaply | Summary errors become persistent context |
| Retrieval | Selects relevant stored facts or past turns | Scales beyond one context window | Poor retrieval can omit or inject irrelevant data |
| Provider-managed state | References a conversation or previous response | Less client-side message plumbing | Still requires retention, privacy, and budget decisions |

A common production design combines them: recent turns verbatim, a carefully maintained summary for older conversation, and retrieved user facts only when relevant. Treat summaries as derived data that can be inspected and rebuilt—not unquestionable truth.


## Failure cases and improvements

| Failure | Why it happens | Improvement |
|---|---|---|
| Context grows forever | Every turn is appended unconditionally | Enforce an input budget before every call |
| System instructions disappear | Naive slicing keeps only the last messages | Pin high-priority instructions separately |
| Assistant reply has no preceding user turn | Individual messages are trimmed | Keep complete exchanges or protocol groups |
| Different users share history | Mutable global history is reused | Store state per session and authorize access |
| Old user text becomes an instruction | Historical content is treated as trusted policy | Preserve role boundaries and never promote user content |
| Token estimate is wrong | Character heuristics vary by language/model | Use provider-supported token counting and safety margin |


## Exercise

1. Lower `max_input_tokens` and identify which complete exchange disappears first.
2. Add a `summary` message that is included after the system instruction but before recent exchanges.
3. Reserve a separate output budget and reject impossible configurations.
4. Add a test proving one `Conversation` instance cannot leak messages into another.
5. Explain which data you would persist, for how long, and how a user could delete it.


## Summary

History is the application record; context is the bounded evidence sent for one model call; memory is durable information stored outside that immediate request. Build context deliberately: preserve instructions, include the newest request, retain complete recent exchanges, retrieve only relevant memory, and reserve room for output. Context management is an application responsibility, not a magical property of chat models.
